In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling import SobolQMCNormalSampler
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.models.transforms import Normalize, Standardize
from botorch.utils.sampling import draw_sobol_samples

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, Normalizer
import umap


# Plot the data
Independent from the BayesOpt model

### PCA / UMAP (Not useful probably)

In [ ]:
# Standardization (z-score) zero mean, unit variance
# computes z = (x - mean) / std
data_standardized = StandardScaler().fit_transform(data[variable_order])

# since liquid1 and liquid2 are on the same scale, we should standardize them together
# compute the mean and std of liquid1 and liquid2
liquids_mean = np.array([data["liquid1"].mean(), data["liquid2"].mean()]).mean()
liquids_std = np.array([data["liquid1"].std(), data["liquid2"].std()]).mean()
# standardize liquid1 and liquid2 by their common mean and std
data_standardized[:, variable_order.index("liquid1")] = (data["liquid1"] - liquids_mean) / liquids_std
data_standardized[:, variable_order.index("liquid2")] = (data["liquid2"] - liquids_mean) / liquids_std

# for each column print the mean and std
# should all be close to 0 and 1, except for the liquids
for i, col in enumerate(variable_order):
    print(f"{col}: mean={data_standardized[:, i].mean()}, std={data_standardized[:, i].std()}")

In [ ]:
# PCA preserves global structure (at least in a linear sense), distances between points are meaningful
# Only captures linear patterns (not good for complex non-linear relationships).
# PCA works by finding directions (principal components) that maximize variance in the data.

# If you apply PCA to just the six input variables (without considering the cost function), 
# you can visualize how the inputs are distributed. I.e. it tells about how the inputs were picked.
# Since most of the data is exploration, the inputs are quite uninformative.
# We could plot the inputs of only the final exploitation experiments.

# Applying PCA to just the input variables,
# will not directly tell you which regions of the input space result in a good cost.
# But we can color the PCA plot by the cost function to see if there are regions that are better or worse.
# PCA does not account for cost while computing components, 
# so there might not be clusters that align perfectly with cost variations.

pca = PCA(n_components=2)
embedding = pca.fit_transform(data_standardized)

# plot the embedding
fig = px.scatter(embedding, x=0, y=1, color=data["stability_slope"], title="PCA of data")
fig.update_layout(
    xaxis_title="PCA 1",
    yaxis_title="PCA 2",
    showlegend=True,
    margin=dict(l=0, r=0, t=30, b=0)  # Remove whitespace around plot
)
fig.write_image(f"{plotfolder}/pca.png")
fig.show()

In [ ]:

# Get PCA loadings (how much each variable contributes to each PC)
pca_loadings = pca.components_
# print the loadings of each variable for each component
for i, loading in enumerate(pca_loadings):
    print(f"Weight for each input variable in PCA{i+1}: {loading}") 

# print out component "formula"
for i, loading in enumerate(pca_loadings):
    pca_formula = ""
    for j, var in enumerate(variable_order):
        pca_formula += f"{loading[j]:.2f} * {var} + "
    pca_formula = pca_formula[:-3]
    print(f"PCA component {i+1} formula: {pca_formula}")

# print the variance of each component
print(f"Variance of each component: {pca.explained_variance_}")
# # print the variance ratio of each component
# print(f"Variance ratio of each component: {pca.explained_variance_ratio_}")
# # print the cumulative variance ratio of each component
# print(f"Cumulative variance ratio of each component: {pca.explained_variance_ratio_.cumsum()}")


In [ ]:
# # UMAP preserves local and global structure approximately
# # allows for non-linear relationships

# myupmap = umap.UMAP(n_components=2, random_state=42)
# embedding = myupmap.fit_transform(data_standardized) 

# # plot the embedding
# fig = px.scatter(embedding, x=0, y=1, color=data["stability_slope"], title="UMAP of data")
# fig.update_layout(
#     xaxis_title="UMAP 1",
#     yaxis_title="UMAP 2",
#     showlegend=True,
#     margin=dict(l=0, r=0, t=30, b=0)  # Remove whitespace around plot
# )
# fig.write_image(f"{plotfolder}/umap.png")
# fig.show()
